# 05 — External Evaluation, Calibration, and Fairness

**Objective:** Evaluate the persisted winner once on the untouched 10% raw holdout and audit probability quality and groups.

In [1]:
from pathlib import Path
import os, sys
import pandas as pd
import plotly.express as px

_cwd = Path.cwd().resolve()
ROOT = _cwd.parent if _cwd.name == "dev" else _cwd
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/credit-risk-lab-matplotlib")

from credit_risk_lab.config.settings import settings
print(f"Project root: {settings.project_root}")

Project root: /Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab


## 1. Score the untouched raw external test through the real inference pipeline

In [2]:
from credit_risk_lab.application import RawLoanScorer
from credit_risk_lab.infrastructure.data_sources import CSVDataSourceConfig, CSVDatasetRepository
from credit_risk_lab.infrastructure.modeling import load_model_bundle

test_raw = CSVDatasetRepository(CSVDataSourceConfig(path=settings.test_path)).load()
bundle = load_model_bundle(settings.model_bundle_path)
scorer = RawLoanScorer(bundle, threshold=settings.decision_threshold)
scored = scorer.score(test_raw.drop(columns=[settings.target_column]))
{"rows": len(scored.probabilities), "model": scored.model_name, "serving_threshold": scored.threshold}

2026-07-11 09:52:31 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:51 - Chargement du fichier : /Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab/data/processed/test.csv


2026-07-11 09:52:31 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:59 - Dataset chargé (4500 lignes, 14 colonnes)


2026-07-11 09:52:31 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Entrée - shape = (4500, 13)


2026-07-11 09:52:31 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Sortie - shape = (4500, 38)


{'rows': 4500, 'model': 'LightGBM', 'serving_threshold': 0.25}

## 2. External metrics at the operational YAML threshold

In [3]:
from credit_risk_lab.infrastructure.evaluation import classification_metrics

y_external = test_raw[settings.target_column].astype(int)
external_metrics = classification_metrics(y_external, scored.probabilities, settings.decision_threshold)
pd.Series(external_metrics).sort_index().to_frame("value")

,value
accuracy,0.909333
balanced_accuracy,0.912429
brier,0.047430
cohen_kappa,0.758739
ece,0.014256
f1,0.818182
gini,0.957135
ks,0.828857
log_loss,0.150227
mcc,0.766762


## 3. Calibration table and curve

In [4]:
from credit_risk_lab.infrastructure.evaluation import calibration_table
from credit_risk_lab.infrastructure.visualization import plot_calibration

calibration = calibration_table(y_external, scored.probabilities, bins=10)
display(calibration.round(4))
plot_calibration(calibration, scored.model_name).show()

,bin,lower_bound,upper_bound,rows,mean_probability,observed_rate,absolute_gap
0,1,0.0,0.1,2786,0.0088,0.0050,0.0037
1,2,0.1,0.2,329,0.1462,0.1064,0.0399
2,3,0.2,0.3,233,0.2450,0.2318,0.0132
3,4,0.3,0.4,159,0.3447,0.3648,0.0201
4,5,0.4,0.5,104,0.4478,0.4327,0.0152
5,6,0.5,0.6,74,0.5474,0.6351,0.0878
6,7,0.6,0.7,67,0.6527,0.5522,0.1005
7,8,0.7,0.8,76,0.7504,0.6842,0.0662
8,9,0.8,0.9,99,0.8507,0.8788,0.0281
9,10,0.9,1.0,573,0.9760,0.9965,0.0205


## 4. Group diagnostics

Sensitive variables were excluded from training and are used here only for governance.

In [5]:
from credit_risk_lab.infrastructure.evaluation import fairness_report

sensitive = test_raw[[c for c in settings.sensitive_columns if c in test_raw]]
fairness = fairness_report(y_external, scored.probabilities, sensitive, settings.decision_threshold)
fairness

,attribute,group,rows,small_group,positive_rate,selection_rate,true_positive_rate,false_positive_rate,mean_probability,observed_rate,calibration_gap,brier
0,person_gender,female,2008,False,0.224602,0.276892,0.917960,0.091201,0.226793,0.224602,0.002191,0.047114
1,person_gender,male,2492,False,0.220305,0.276083,0.918033,0.094699,0.224847,0.220305,0.004542,0.047684


## 5. Persist external evaluation

In [6]:
external_path = settings.reports_dir / "external_test_metrics.csv"
pd.DataFrame([{"model": scored.model_name, "threshold": settings.decision_threshold, **external_metrics}]).to_csv(external_path, index=False)
external_path

PosixPath('/Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab/reports/external_test_metrics.csv')